# Notebook for finding the parameters where recall is 1 and runtime is good

In [69]:
import os
import sys
import itertools


def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from utils.helpers.measure_similarities import *

Project root found: c:\Users\eivin\dev\masteroppgave


## Read Bucketing Runtime CSV

In [70]:
CITY = "rome"
BUCKETING_METHOD = "loose"
MEASURE = "dtw"
DATA_SIZE = [500]
SCHEME = "grid"


folder_path = f"../../results_hashed/runtimes/bucketing/{CITY}/true_trajectories/{BUCKETING_METHOD}/{MEASURE}/"
file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(true_trajectories)_{CITY}_{MEASURE}_{DATA_SIZE}_{SCHEME}.csv"
file_path = folder_path + file_name

runtime_bucketing = pd.read_csv(file_path, index_col=0)
runtime_bucketing.head()


,Measure,Resolution,Layers,Size,Average Similarity Computation Time (Seconds),Average Hash Generation Time (Seconds),Average Bucket Distribution Time (Seconds),Total time (Seconds)
City,,,,,,,,
rome,dtw,0.1,1,500,14.297,3.038,0.108,17.443
rome,dtw,0.1,2,500,14.593,5.296,0.239,20.128
rome,dtw,0.6,2,500,27.867,0.912,0.038,28.817
rome,dtw,0.1,3,500,14.889,6.920,0.333,22.142
rome,dtw,1.1,2,500,41.417,0.906,0.011,42.334


## Read No Bucketing Runtime CSV

In [71]:
CITY = "porto"
BUCKETING_METHOD = "loose"
MEASURE = "dtw"
DATA_SIZE = [500]
SCHEME = "grid"


folder_path = f"../../results_hashed/runtimes/no_bucketing/{CITY}/{MEASURE}/"
file_name = f"runtimes_{CITY}_{MEASURE}_{DATA_SIZE}_{SCHEME}_no_bucketing.csv"
file_path = folder_path + file_name

runtime_no_bucketing = pd.read_csv(file_path, index_col=0)

## Read Bucket Evaluation CSV

In [73]:
CITY = "porto"
BUCKETING_METHOD = "loose"
MEASURE = "dtw"
DATA_SIZE = 500
SCHEME = "grid"

folder_path = f"../../results_hashed/bucket_evaluation/{BUCKETING_METHOD}/"
file_name = f"{CITY}_{MEASURE}_{SCHEME}_{DATA_SIZE}.csv"
file_path = folder_path + file_name

bucket_evaluation = pd.read_csv(file_path, index_col=0)

In [67]:
# First, filter the bucket_evaluation dataframe
filtered_eval = bucket_evaluation[
    (bucket_evaluation["Avg Recall"] == 1) &
    (bucket_evaluation["Threshold"] == 0.1)
]

# Select only the necessary columns from runtime_bucketing
runtime_selected = runtime_bucketing[[
    "Resolution", "Layers",
    "Average Similarity Computation Time (Seconds)",
    "Average Hash Generation Time (Seconds)",
    "Average Bucket Distribution Time (Seconds)",
    "Total time (Seconds)"
]]

# Merge using only Diameter, Layers, and Disks as keys
merged_df = filtered_eval.merge(
    runtime_selected,
    on=["Resolution", "Layers"],
    how="left"
)

merged_df.head(10)

,Measure,Resolution,Layers,Size,Threshold,Avg Precision,Avg Recall,Avg F1 Score,Avg Total Buckets,Avg Largest Bucket Size,Avg Smallest Bucket Size,Avg Buckets with >1 Trajectory,Avg Buckets with 1 Trajectory,Avg Percentage >1 Trajectory,Avg Percentage 1 Trajectory,Avg Correlation Coefficient,Average Similarity Computation Time (Seconds),Average Hash Generation Time (Seconds),Average Bucket Distribution Time (Seconds),Total time (Seconds)
0,dtw,0.1,1,500,0.1,0.001,1.0,0.002,2187.583,114.208,1.0,1668.583,519.000,76.277,23.723,0.541,14.297,3.038,0.108,17.443
1,dtw,0.1,2,500,0.1,0.001,1.0,0.002,4368.917,124.167,1.0,3336.000,1032.917,76.358,23.642,0.542,14.593,5.296,0.239,20.128
2,dtw,0.1,3,500,0.1,0.001,1.0,0.002,6555.625,128.917,1.0,4998.042,1557.583,76.241,23.759,0.544,14.889,6.920,0.333,22.142
3,dtw,0.1,4,500,0.1,0.001,1.0,0.002,8736.375,131.250,1.0,6655.292,2081.083,76.180,23.820,0.547,15.282,7.094,0.289,22.665
4,dtw,0.1,5,500,0.1,0.001,1.0,0.002,10915.458,131.875,1.0,8322.750,2592.708,76.248,23.752,0.545,15.699,10.347,0.596,26.642
5,dtw,0.1,6,500,0.1,0.001,1.0,0.002,13106.208,130.708,1.0,9993.500,3112.708,76.250,23.750,0.548,16.090,13.548,1.368,31.006
6,dtw,0.2,1,500,0.1,0.001,1.0,0.002,792.917,140.417,1.0,699.083,93.833,88.172,11.828,0.461,15.850,1.608,0.033,17.491
7,dtw,0.2,2,500,0.1,0.001,1.0,0.002,1585.583,145.083,1.0,1405.375,180.208,88.637,11.363,0.474,15.973,3.314,0.163,19.450
8,dtw,0.2,3,500,0.1,0.001,1.0,0.002,2380.375,145.792,1.0,2105.792,274.583,88.466,11.534,0.478,16.300,2.944,0.182,19.426
9,dtw,0.2,4,500,0.1,0.001,1.0,0.002,3171.042,147.333,1.0,2805.417,365.625,88.471,11.529,0.484,16.321,4.198,0.300,20.819


In [74]:
# Sort the merged dataframe by Avg Correlation Coefficient in descending order
merged_df_sorted = merged_df.sort_values(
    by="Avg Correlation Coefficient",
    ascending=False
)
merged_df_sorted.head(10)



,Measure,Resolution,Layers,Size,Threshold,Avg Precision,Avg Recall,Avg F1 Score,Avg Total Buckets,Avg Largest Bucket Size,Avg Smallest Bucket Size,Avg Buckets with >1 Trajectory,Avg Buckets with 1 Trajectory,Avg Percentage >1 Trajectory,Avg Percentage 1 Trajectory,Avg Correlation Coefficient,Average Similarity Computation Time (Seconds),Average Hash Generation Time (Seconds),Average Bucket Distribution Time (Seconds),Total time (Seconds)
5,dtw,0.1,6,500,0.1,0.001,1.0,0.002,13106.208,130.708,1.0,9993.500,3112.708,76.250,23.750,0.548,16.090,13.548,1.368,31.006
3,dtw,0.1,4,500,0.1,0.001,1.0,0.002,8736.375,131.250,1.0,6655.292,2081.083,76.180,23.820,0.547,15.282,7.094,0.289,22.665
4,dtw,0.1,5,500,0.1,0.001,1.0,0.002,10915.458,131.875,1.0,8322.750,2592.708,76.248,23.752,0.545,15.699,10.347,0.596,26.642
2,dtw,0.1,3,500,0.1,0.001,1.0,0.002,6555.625,128.917,1.0,4998.042,1557.583,76.241,23.759,0.544,14.889,6.920,0.333,22.142
1,dtw,0.1,2,500,0.1,0.001,1.0,0.002,4368.917,124.167,1.0,3336.000,1032.917,76.358,23.642,0.542,14.593,5.296,0.239,20.128
0,dtw,0.1,1,500,0.1,0.001,1.0,0.002,2187.583,114.208,1.0,1668.583,519.000,76.277,23.723,0.541,14.297,3.038,0.108,17.443
80,dtw,1.0,5,500,0.1,0.001,1.0,0.001,175.000,247.542,2.0,175.000,0.000,100.000,0.000,0.488,37.353,2.546,0.067,39.966
79,dtw,1.0,5,500,0.1,0.001,1.0,0.001,175.000,247.542,2.0,175.000,0.000,100.000,0.000,0.488,37.600,2.524,0.064,40.188
99,dtw,1.1,6,500,0.1,0.000,1.0,0.001,180.000,267.125,2.0,180.000,0.000,100.000,0.000,0.486,38.857,2.473,0.062,41.392
97,dtw,1.1,6,500,0.1,0.000,1.0,0.001,180.000,267.125,2.0,180.000,0.000,100.000,0.000,0.486,41.213,2.284,0.058,43.555
